In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm

from transformers import AutoModelForCausalLM
from time_moe.models.modeling_time_moe import TimeMoeForPrediction, TimeMoeConfig
from time_moe.trainer.hf_trainer import TimeMoETrainingArguments, TimeMoeTrainer
from run_eval import TimeMoE, BenchmarkEvalDataset, MSEMetric, MAEMetric, DataLoader, evaluate
import argparse

import warnings
warnings.filterwarnings('ignore')

In [2]:
%load_ext autoreload
%autoreload 2

In [5]:
# 1. Load pretrained TimeMoE
model1 = TimeMoE(
    'Maple728/TimeMoE-50M',
    'cpu',
    context_length=192,
    prediction_length=4,
)

device = torch.device("cpu")


batch_size = 32
context_length = 192

`torch_dtype` is deprecated! Use `dtype` instead!


In [3]:
parser = argparse.ArgumentParser('TimeMoE Evaluate')
args = parser.parse_args(args=[])


args.model = 'Maple728/TimeMoE-50M'
args.batch_size = batch_size
args.context_length = context_length
args.prediction_length = 96
args.data = 'dataset/AAPL_prices.csv'

evaluate(args)

2025-11-03 22:27:06,114 - log_util.py[pid:24129;line:52:log_in_local_rank_0] - INFO: >>> Split test data from 2022-11-01 to 2024-12-31, and evaluation start date is: 2023-08-09


  0%|          | 0/40 [00:11<?, ?it/s]


AttributeError: 'DynamicCache' object has no attribute 'seen_tokens'

In [7]:
import os
import json
import torch
import pandas as pd
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
import numpy as np
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split



In [8]:
def setup_lora(model, lora_config=None):
    """Setup LoRA configuration for the model"""
    
    if lora_config is None:
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,  # Adjust based on your task
            inference_mode=False,
            r=8,  # Rank
            lora_alpha=32,
            lora_dropout=0.1,
            target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]  # Adjust for your model
        )
    
    lora_model = get_peft_model(model, lora_config)
    lora_model.print_trainable_parameters()
    
    return lora_model

In [14]:
train_dataset = BenchmarkEvalDataset(
    'dataset/AAPL_prices.csv',
    context_length=720,
    prediction_length=72,
)

2025-11-04 08:55:12,127 - log_util.py[pid:24129;line:52:log_in_local_rank_0] - INFO: >>> Split test data from 2020-09-28 to 2024-12-31, and evaluation start date is: 2023-08-09


In [ ]:
def fine_tune_with_lora_simple(model_name, train_dataset):
    """Simplified fine-tuning function"""
    
    # Detect available device
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    # Load model
    model = TimeMoE(
        'Maple728/TimeMoE-50M',
        device,
        context_length=1024,
        prediction_length=192,
    )
    
    # Setup LoRA
    lora_model = setup_lora(model.model)
    
    # Only enable fp16 if using CUDA
    use_fp16 = torch.cuda.is_available()
    
    # Simple training arguments - no load_best_model_at_end to avoid conflicts
    training_args = TrainingArguments(
        output_dir="./lora_finetuned_timemoe",
        overwrite_output_dir=True,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=2,
        learning_rate=1e-4,
        num_train_epochs=3,
        logging_dir="./logs",
        logging_steps=10,
        warmup_steps=50,
        fp16=use_fp16,
        dataloader_pin_memory=False,
        remove_unused_columns=False,
        report_to=None,  # Disable wandb/tensorboard if not needed
    )
    
    print(f"Training with fp16: {use_fp16}")
    print(f"Train dataset size: {len(train_dataset)}")
    
    # Simple data collator
    def simple_data_collator(features):
        if not features:
            return {}
        
        # Get the first feature to understand structure
        first_feature = features[0]
        print(f"Data collator processing features with keys: {list(first_feature.keys())}")
        
        # Try to find numerical data
        for key, value in first_feature.items():
            if isinstance(value, (int, float)):
                # Single numerical values
                values = [f[key] for f in features]
                batch = {
                    "input_ids": torch.tensor(values, dtype=torch.float32).unsqueeze(1),
                    "labels": torch.tensor(values, dtype=torch.float32).unsqueeze(1)
                }
                return batch
            elif isinstance(value, list) and all(isinstance(x, (int, float)) for x in value):
                # List of numerical values
                values = [f[key] for f in features]
                batch = {
                    "input_ids": torch.tensor(values, dtype=torch.float32),
                    "labels": torch.tensor(values, dtype=torch.float32)
                }
                return batch
        
        # Fallback: create simple dummy data
        print("Using fallback dummy data")
        batch = {
            "input_ids": torch.randn(len(features), 10),
            "labels": torch.randn(len(features), 4)
        }
        return batch
    
    # Create trainer
    trainer = TimeMoeTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
    )
    
    # Start training
    print("Starting training...")
    trainer.train()
    
    # Save model
    trainer.save_model()
    print("Model saved successfully!")
    
    return trainer, lora_model

# Use the simplified version
print("\nStep 3: Starting fine-tuning...")
trainer, model = fine_tune_with_lora_simple("TimeMoe", train_dataset)


Step 3: Starting fine-tuning...
Using device: cpu


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: a81be3d3-3f3e-476f-9376-9fe282e276f1)')' thrown while requesting HEAD https://huggingface.co/Maple728/TimeMoE-50M/resolve/main/config.json
Retrying in 1s [Retry 1/5].


trainable params: 294,912 || all params: 113,647,104 || trainable%: 0.25949803349146494
Training with fp16: False
Train dataset size: 1400
Starting training...


TypeError: TimeMoeForPrediction.forward() got an unexpected keyword argument 'inputs'

In [9]:
from time_moe.runner import TimeMoeRunner

runner = TimeMoeRunner()


model = TimeMoE(
    'Maple728/TimeMoE-50M',
    device,
    context_length=1024,
    prediction_length=192,
)

# Setup LoRA
lora_model = setup_lora(model.model)

model = runner.train_model(model=lora_model)


trainable params: 294,912 || all params: 113,647,104 || trainable%: 0.25949803349146494
2025-11-04 09:15:35,610 - log_util.py[pid:28524;line:52:log_in_local_rank_0] - INFO: Set global_batch_size to 16
2025-11-04 09:15:35,610 - log_util.py[pid:28524;line:52:log_in_local_rank_0] - INFO: Set micro_batch_size to 16
2025-11-04 09:15:35,610 - log_util.py[pid:28524;line:52:log_in_local_rank_0] - INFO: Set gradient_accumulation_steps to 1
2025-11-04 09:15:35,611 - log_util.py[pid:28524;line:52:log_in_local_rank_0] - INFO: Set precision to bf16


KeyError: 'normalization_method'

## OMG

In [13]:
import pandas as pd
import numpy as np
import json

# Load your CSV
df = pd.read_csv("dataset/AAPL_prices.csv", parse_dates=["date"])
df = df.sort_values("date")

# Compute returns (or you may choose another target variable)
df["return"] = np.log(df["Close"] / df["Close"].shift(1))
df = df.dropna().reset_index(drop=True)

# Decide context + prediction length
context_len = 128
pred_len = 16
seq_len = context_len + pred_len

jsonl_path = "appl_time_moe_dataset.jsonl"
with open(jsonl_path, "w") as f:
    # Slide window over your return series
    for start in range(0, len(df) - seq_len + 1):
        subseq = df["return"].iloc[start : start + seq_len].tolist()
        obj = {"sequence": subseq}
        f.write(json.dumps(obj) + "\n")

print("Wrote", jsonl_path)


Wrote appl_time_moe_dataset.jsonl


In [14]:
from opik import track

@track
def my_function(input: str) -> str:
    return input


[autoreload of transformers.utils.import_utils failed: Traceback (most recent call last):
  File "/Users/naburkova/repos/project/venv312/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/Users/naburkova/repos/project/venv312/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 621, in superreload
    update_generic(old_obj, new_obj)
  File "/Users/naburkova/repos/project/venv312/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 447, in update_generic
    update(a, b)
  File "/Users/naburkova/repos/project/venv312/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 415, in update_class
    update_instances(old, new)
  File "/Users/naburkova/repos/project/venv312/lib/python3.12/site-packages/IPython/extensions/autoreload.py", line 373, in update_instances
    object.__setattr__(ref, "__class__", new)
TypeError: can't apply this __setattr__ to DummyObj